# 🧠 `<think>` CoT × base比較 — True-LoRA Notebook

プロンプトに **`<think>` タグ** を付けて matutake に **Chain-of-Thought（思考過程）** をさせ、
さらに **True-LoRA で生成したアダプタ** を当てた場合と **base（素のmatutake）** を **同じモデル上で切り替えて比較** します。

- **Part A**: プロンプトだけで `<think>…</think>` CoT（LoRA不要）
- **Part B**: True-LoRA を生成 → `temporary_lora` で **on/off を切り替えて base と比較**
- **（高度）**: end-to-end SFT で `<think>` 傾向を実際に強める

> 生成は **CPUで数ms**。推論は実モデル（2B）を読むので **GPU推奨**。
> 比較は1つのモデルを読み込み、LoRAを一時適用→自動復元するのでメモリは1モデル分だけです。

## Step 1: セットアップ

In [ ]:
!pip install git+https://github.com/MARVserver/TrueLora.git -q
!pip install "sentence-transformers" transformers accelerate -q
print("Done!")

In [ ]:
import re
from pathlib import Path

import torch
from true_lora.adapter import AdapterSpec, LoraTensorSpec, save_peft_adapter
from true_lora.generator import TrueLoraGenerator
from true_lora.text import SemanticTextEncoder
from true_lora.train import train_on_adapter_bank
from true_lora.apply import temporary_lora

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
OUTPUT_DIR = Path("./output")
OUTPUT_DIR.mkdir(exist_ok=True)
MODEL_NAME = "summerMC/matutake"
print(f"Device: {DEVICE}")

## Part A: プロンプトで `<think>` CoT をさせる

CoT は **プロンプト（system指示）** で誘導します。モデルに「`<think>…</think>` の中で順を追って考え、
そのあとに最終回答を書く」よう指示し、出力を `<think>` ブロックと最終回答に分けて表示します。

In [ ]:
THINK_SYSTEM = (
    "You are a careful reasoner. First think step by step INSIDE "
    "<think> </think> tags. After </think>, give a concise final answer. "
    "日本語の質問には日本語で答えてください。"
)

def build_messages(question: str) -> list[dict]:
    return [
        {"role": "system", "content": THINK_SYSTEM},
        {"role": "user", "content": question},
    ]

def split_think(text: str) -> tuple[str, str]:
    """Return (think_reasoning, final_answer). Falls back gracefully if no tags."""
    m = re.search(r"<think>(.*?)</think>(.*)", text, flags=re.DOTALL)
    if m:
        return m.group(1).strip(), m.group(2).strip()
    # No closing tag: treat everything after a lone <think> as reasoning.
    if "<think>" in text:
        return text.split("<think>", 1)[1].strip(), ""
    return "", text.strip()

def show_reply(tag: str, text: str) -> None:
    think, answer = split_think(text)
    print(f"=== {tag} ===")
    if think:
        print("  <think>:", (think[:400] + ("…" if len(think) > 400 else "")))
    print("  answer :", (answer or text)[:400])
    print(f"  has <think> tag: {'<think>' in text}")

print("CoTプロンプト準備完了")

## Part B-1: matutake 用の LoRA を生成（bankless）

`adapter_bank=None` の純 Text-to-LoRA。matutake は GQA なので q_proj と v_proj の出力次元が異なります。
学習データには「推論・`<think>` CoT」系の説明文を含め、ハイパーネットに学習させます。

In [ ]:
LORA_RANK, LORA_ALPHA, MAX_NORM = 4, 8.0, 4.0

from transformers import AutoConfig
cfg = AutoConfig.from_pretrained(MODEL_NAME, trust_remote_code=True)
HIDDEN = cfg.hidden_size
N_HEADS = cfg.num_attention_heads
N_KV = getattr(cfg, "num_key_value_heads", N_HEADS)
HEAD_DIM = getattr(cfg, "head_dim", HIDDEN // N_HEADS)
PROJ_OUT = {"q_proj": N_HEADS * HEAD_DIM, "v_proj": N_KV * HEAD_DIM}
TARGETS, LAYERS = ["q_proj", "v_proj"], [0, 6, 12, 18]

lora_specs = []
for li in LAYERS:
    for t in TARGETS:
        name = f"model.layers.{li}.self_attn.{t}"
        lora_specs.append(LoraTensorSpec(name, PROJ_OUT[t], HIDDEN, LORA_RANK, LORA_ALPHA))
print(f"hidden={HIDDEN} specs={len(lora_specs)} out_features={PROJ_OUT}")

In [ ]:
encoder = SemanticTextEncoder()
print(f"Encoder: {encoder.backend} dim={encoder.dim}")

# (説明文, lora_A値, lora_B値) — <think> CoT / 推論系を厚めに
train_config = [
    ("step-by-step chain-of-thought reasoning with <think> tags", 0.22, 0.11),
    ("think before answering, show reasoning inside think tags", 0.21, 0.10),
    ("careful logical deduction and multi-step reasoning", 0.20, 0.10),
    ("math word problem solving with intermediate steps", 0.23, 0.11),
    ("explain the reasoning then give the final answer", 0.19, 0.09),
    ("日本語で順を追って考えてから答える", 0.18, 0.09),
    ("python code generation", 0.15, 0.07),
    ("creative storytelling", 0.13, 0.06),
    ("japanese translation", 0.12, 0.06),
    ("concise factual question answering", 0.11, 0.05),
]
adapters = []
for desc, sa, sb in train_config:
    tensors = {}
    for sp in lora_specs:
        tensors[f"{sp.name}.lora_A.weight"] = torch.full(sp.a_shape, sa)
        tensors[f"{sp.name}.lora_B.weight"] = torch.full(sp.b_shape, sb)
    adapters.append(AdapterSpec(desc, encoder.encode(desc), tensors, metrics={"score": 0.8}))

generator = TrueLoraGenerator(
    tensor_specs=lora_specs, adapter_bank=None, hidden_dim=256,
    max_tensor_norm=MAX_NORM, encoder=encoder, hyper_kind="conditioned",
)
losses = train_on_adapter_bank(generator, adapters, steps=200, lr=1e-3)
# seen プロンプトを anchor 登録 → 未知プロンプトでは確信度が下がる（novelty信号）
generator.set_distribution_anchors(adapters)
print(f"trained loss {losses[0]:.4f} -> {losses[-1]:.6f}")

In [ ]:
def make_cot_lora(prompt: str, save: bool = False):
    """プロンプトから <think> CoT 用 LoRA を生成。confidence も表示。"""
    state_dict, report = generator.generate(prompt, retrieval_k=8)
    conf = 1.0 - report["uncertainty"]
    print(f"LoRA generated for: {prompt!r}  | confidence={conf:.1%} "
          f"(novelty={report.get('novelty', 0.0):.2f})")
    path = None
    if save:
        path = OUTPUT_DIR / "cot_think_lora.pt"
        save_peft_adapter(path, {k: v.cpu() for k, v in state_dict.items()}, report)
        print(f"  saved: {path}")
    return state_dict, report, path

cot_state, cot_report, _ = make_cot_lora(
    "step-by-step chain-of-thought reasoning with <think> tags", save=True
)

## Part B-2: matutake を読み込む（GPU推奨）

比較は **重みのマージ→復元** で行うため、量子化なしの実重み（bf16 / fp32）で読み込みます。
（4bit量子化だと `nn.Linear` ではなくなり一時マージできません。）

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

dtype = torch.bfloat16 if DEVICE == "cuda" else torch.float32
print("Loading matutake (this is the only model we load)...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, trust_remote_code=True, torch_dtype=dtype,
).to(DEVICE)
model.eval()
print("Loaded!")

In [ ]:
@torch.no_grad()
def generate_reply(messages, max_new_tokens=256):
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    out = model.generate(
        **inputs, max_new_tokens=max_new_tokens,
        do_sample=False,  # greedy: base vs adapted の差を LoRA だけに帰属させる
        pad_token_id=tokenizer.eos_token_id,
    )
    return tokenizer.decode(out[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True)

print("generate_reply 準備完了")

## Part A 実行: プロンプトだけで `<think>` CoT（base, LoRAなし）

In [ ]:
QUESTIONS = [
    "If a train travels 60 km in 1.5 hours, what is its average speed?",
    "りんごが3個、みかんが5個あります。果物は全部で何個ですか？",
    "What is the next number in the sequence 2, 6, 12, 20, ?",
]

for q in QUESTIONS:
    reply = generate_reply(build_messages(q))
    print("\nQ:", q)
    show_reply("BASE (prompt CoT)", reply)
    print("-" * 72)

## Part B 実行: base vs adapted を比較（同一プロンプト・LoRAをon/off）

プロンプトは固定（両方とも `<think>` 指示つき）。**LoRA を一時適用するかどうかだけ** を変えるので、
差分は生成アダプタの効果に帰属します。`temporary_lora` がブロックを抜けると重みを自動復元します。

In [ ]:
cot_sd = cot_state  # Part B-1 で生成した <think> CoT 用 LoRA

for q in QUESTIONS:
    msgs = build_messages(q)
    base_reply = generate_reply(msgs)
    with temporary_lora(model, cot_sd, lora_specs, strict=False) as applied:
        lora_reply = generate_reply(msgs)
    print("\nQ:", q, f"  (LoRA applied to {len(applied)} modules)")
    show_reply("BASE   ", base_reply)
    show_reply("ADAPTED", lora_reply)
    print("=" * 72)

## （高度・オプション）end-to-end SFT で `<think>` を実際に強める

上の再構成学習 LoRA は軽いnudgeです。**実際に `<think>` を出すよう** 学習させるには end-to-end SFT:
matutake に LoRA を当て、`<think>…</think>` を含む教師ラベルの causal LM loss を **ハイパーネットまで逆伝播** します。
（実モデルへ逆伝播するため重め。`<think>` 付きの教師データを増やすほど効果が出ます。）

In [ ]:
# 注意: 実モデルへの逆伝播のため GPU 推奨・時間がかかります。お試し用に steps は小さめ。
from true_lora.sft import sft_train_hypernetwork, causal_lm_loss

def think_batch(question: str, think: str, answer: str) -> dict:
    msgs = build_messages(question)
    prompt_text = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    target = f"<think>{think}</think> {answer}{tokenizer.eos_token}"
    full = tokenizer(prompt_text + target, return_tensors="pt")
    prompt_len = tokenizer(prompt_text, return_tensors="pt")["input_ids"].shape[1]
    labels = full["input_ids"].clone()
    labels[:, :prompt_len] = -100  # プロンプト部分は loss に含めない
    return {k: v.to(model.device) for k, v in full.items()} | {"labels": labels.to(model.device)}

examples = [
    ("step-by-step reasoning with <think> tags",
     think_batch("What is 12 x 8?", "12 times 8 is 96.", "96")),
    ("think before answering",
     think_batch("Is 17 a prime number?",
                 "17 has no divisors other than 1 and 17.", "Yes, 17 is prime.")),
]

sft_losses = sft_train_hypernetwork(
    generator, model, examples, causal_lm_loss, steps=30, lr=1e-4,
)
print(f"SFT loss {sft_losses[0]:.4f} -> {sft_losses[-1]:.4f}")

# SFT後の LoRA で再度 base と比較
cot_sd2, _report = generator.generate("step-by-step reasoning with <think> tags")
for q in QUESTIONS[:1]:
    msgs = build_messages(q)
    with temporary_lora(model, cot_sd2, lora_specs, strict=False):
        print("\nQ:", q)
        show_reply("ADAPTED (after SFT)", generate_reply(msgs))

## まとめ

| 項目 | 内容 |
| --- | --- |
| CoTの出し方 | プロンプト（`<think>…</think>` 指示） |
| LoRA生成 | True-LoRA bankless / conditioned（CPUで数ms） |
| 比較方法 | `temporary_lora` で1モデルにon/off（重み自動復元） |
| 差分の帰属 | greedy生成 + 同一プロンプトなので差はLoRAの効果 |
| 強化 | end-to-end SFT（`<think>`付き教師でハイパーネットを学習） |

- `make_cot_lora("好きな推論タスク")` で別の CoT LoRA を生成できます。
- `<think>` の構造はプロンプト由来、LoRA は q_proj / v_proj をシフトして傾向を補強します。
- より強い効果が必要なら SFT セルの教師データを増やしてください。